In [11]:
# 1) Install to your user site with THIS interpreter
#!export PATH="$HOME/.local/bin:$PATH" && pip install --user pandas geopandas networkx matplotlib numpy osmnx shapely

In [12]:
#!bash -lc 'echo '\''export PYTHONPATH="$HOME/.local/lib/python3.13/site-packages:$PYTHONPATH"'\'' >> ~/.bashrc && source ~/.bashrc && echo $PYTHONPATH'

In [13]:
import sys
sys.path.append("/home/WUR/katon002/.local/lib/python3.13/site-packages")

In [14]:
import gurobipy as gp
from gurobipy import GRB
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
from shapely.geometry import Point, LineString
import os
from gurobipy import quicksum
from pathlib import Path
import os, shutil
import gc, sys
import pandas as pd

In [15]:
batteryCost ={
    "ID"        : [ 'Small',  'medium', 'large'],
    "lower(kWh)": [   1,        101,      1001],
    "upper(kWh)": [   100,       500,     7000],
    "Cost/kWh"  : [   520,       420,      400]  
}
# batteryCost ={
#     "ID"        : [ 'Small',  'medium', 'large'],
#     "lower(kWh)": [   1,        101,      1001],
#     "upper(kWh)": [   100,      1000,     1e6],
#     "Cost/kWh"  : [   0.5,       0.4,       0.2]  
# }
df_Batcost =pd.DataFrame(batteryCost)
df5_Batcost= df_Batcost[["ID", "lower(kWh)", "upper(kWh)", "Cost/kWh"]]

In [16]:
#Demand data
# Demand_EV = {
#     "UniqueID"       : [4013, 4962, 7323, 9752, 10445, 5494, 11286, 11744, 12863, 13967, 14257, 139, 4, 10829, 12734, 10837, 292, 517, 1338, 816, 849, 956, 1343, 1457],
#     "POINT_X"        : [162702.5959, 160806.7076, 163137.5767, 163808.8194, 159304.1657, 160847.2871, 161458.1507, 162023.7627, 158564.5389, 162586.3649, 159885.7536, 160575.478, 161546.1922, 162175.9328, 158131.2171, 160295.2478, 158071.1464, 163881.3241, 161683.7643, 162931.5738, 156259.0608, 161346.8151, 158468.292, 161887.7301],
#     "POINT_Y"        : [388343.1643, 384430.2598, 389183.8419, 381057.672, 384714.9847, 380556.4307, 384557.4721, 382560.5136, 381496.1436, 381541.3364, 381695.8029, 387035.874, 385134.076, 382019.7935, 383826.4788, 383584.8842, 387540.4075, 385516.7641, 386983.5747, 381679.7145, 383592.9226, 381981.2913, 388589.9954, 380129.0916],
#     "Population"     : [122.7000931, 22.53254502, 4.941334742, 123.374393, 60.70779358, 1.442556602, 15.39575384, 23.28579294, 109.7481401, 11.92150725, 33.00616403, 46.43129092, 26.3520007, 20.53147131, 70.22421862, 58.1324635, 247.9302098, 41.20232361, 53.1136651, 36.63094804, 38.7019138, 12.90018456, 207.4113305, 89.54674616],
#     "Solar_Supply_MW": [2.8322, 0.5236, 1.2376, 0.9044, 0.7378, 0.3332, 0.5236, 1.5946, 4.9504, 1.071, 1.428, 1.2614, 2.9512, 0.357, 4.9504, 0.4284, 3.1892, 3.57, 1.9516, 1.071, 3.0464, 0.833, 35.6704, 0.4998] 
# }
Demand_EV  =  pd.read_csv("EV_Demand_2024_eindhoven.csv") #"EV_Demand_2024_eindhoven.csv")
Demand = pd.DataFrame(Demand_EV)

EV_proportion = {
    'Models'               : ['Model Y', 'Model 3', 'EX30', 'XC40', 'Golf', 'ID.4', 'ID.3', 'Aygo X', 'Nero', 'Picanto', 'Captur', 'iX1', 'i4', 'Kona', 'Ioniq 5', '208', 'e-2008', 'Kodiaq'],
    '% Proportion'         : [0.171204758, 0.096139853, 0.097038188, 0.060592722, 0.024255055, 0.01454405, 0.016825822, 0.071220029, 0.098044324, 0.075190672, 0.029618118, 0.034792529, 0.018424859, 0.061976158, 0.006503948, 0.054843375, 0.0365892, 0.03219634],
    'Battery Capacity(kWh)': [60, 57.5, 69, 82, 32, 62, 45, 62, 64.8, 58.3, 9.8, 64.8, 68.7, 64.8, 63, 48.1, 54, 25.7],
    'EV Range(km)'         : [393, 513, 403, 472, 324, 336, 330, 336, 463, 429, 375, 333, 418, 420, 394, 410, 400, 100]
        }
df_EV_proportion = pd.DataFrame(EV_proportion)

# Calculate total battery capacity
Demand['Demand(kWh)'] = 0.0
for index, row in Demand.iterrows():
    total_capacity = 0.0
    population = row['Population']
    for _, ev_row in df_EV_proportion.iterrows():
        proportion = ev_row['% Proportion']
        battery_capacity = ev_row['Battery Capacity(kWh)'] #convert to MWh
        capacity = population * proportion * battery_capacity
        total_capacity += capacity
    Demand.at[index, 'Demand(kWh)'] = total_capacity

df1_demand = Demand[["UniqueID", "POINT_X", "POINT_Y", "Population", "Demand(kWh)", "Solar_Supply_MW"]]

In [17]:
EC = {}# units are kWh/km
for v in range(len(df_EV_proportion)):
    EV_model = df_EV_proportion["Models"].iloc[v]
    capacity = (df_EV_proportion["Battery Capacity(kWh)"].iloc[v])/1000 #convert battery capacity to MWh
    ev_range = df_EV_proportion["EV Range(km)"].iloc[v]
    EC[EV_model] = capacity / ev_range

In [18]:
av_EC = sum(EC.values()) / len(EC) # the average energy consumed to move a km.

In [19]:
#Charging station data.
# datacharger = {
#     'ID_unique'                             : [3516, 9327, 12309, 3613, 9329, 3518, 12647, 14960, 24046, 8391],
#     'DependeVar'                            : [1, 1, 1, 1, 1, 1, 1, 0, 0, 1],
#     'POINT_X'                               : [156876.4827, 160013.3214, 158918.6715, 155673.6847, 159494.0311, 156888.2119, 155638.1638, 163714.2396, 157718.7921, 157171.954],
#     'POINT_Y'                               : [383419.989, 379813.1895, 380737.7976, 385205.5304, 380139.7906, 384263.4887, 385295.9729, 380840.1225, 387696.0504, 383443.493],
#     'LandPrice'                             : [425.980011, 456.3009338, 366.0549927, 719.0772705, 419.8666382, 392.0378418, 717.065979, 824.2540894, 374, 491.4030151],
#     'Available Grid intake(Feed-in)(MW)'    : [64, 64, 64, 109, 64, 64, 109, 190, 109, 64],
#     'Grid_supply'                           : [75, 77, 77, 111, 77, 77, 111, 207, 111, 77],
#     'Possible Location feed-in to Grid(MW)' : [137.9110689, 137.9110689, 137.9110689, 128.9965795, 137.9110689, 137.9110689, 128.9965795, 290.6023465, 128.9965795, 137.9110689],
#     'Actual required at location(MW)'       : [110.1706551, 110.1706551, 110.1706551, 121.8170066, 110.1706551, 110.1706551, 121.8170066, 197.6820818, 121.8170066, 110.1706551],
#     'Congestion(MW)'                        : [33, 33, 33, 11, 33, 33, 11, -9, 11, 33],
#     'Feed_in_congestion(MW)'                : [74, 74, 74, 20, 74, 74, 20, 101, 20, 74]
# }
datacharger = pd.read_csv("Chargers_with_Probability_Greater_than_0.5.csv")
df_chargers = pd.DataFrame(datacharger)
df_chargers = df_chargers[df_chargers["Probability"] > 0.5]
df2_chargers = df_chargers[['ID_unique', 'LandPrice', 'Grid_supply', 'Actual required at location(MW)', 'Available Grid intake(Feed-in)(MW)']]
df2_chargers = df2_chargers.reset_index(drop=True)  #reset index after filtering

In [20]:
print(len(df2_chargers))

2033


In [21]:
# Time and month data
time_data = {
    "Time"                  : ["0:00–7:00", "7:00–11:00", "11:00–17:00", "17:00–24:00"],
    "Daily Yield"           : [0.113, 0.503, 0.444, 0.095],
    "Daily Demand variation": [0.096, 0.278, 0.441, 0.185],
    "time_interval"         : [7, 4, 6, 7],
    "Daily tariff(€/kWh)"   : [0.25, 0.32, 0.22, 0.34] #Euro per kWh
}
df3 = pd.DataFrame(time_data)
df3_time = df3[["Time", "Daily Yield", "time_interval", "Daily Demand variation", "Daily tariff(€/kWh)"]]

In [22]:
# month_data = {
#     "Month": ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"],
#     "% Monthly yield": [0.035, 0.051, 0.089, 0.118, 0.124, 0.122, 0.12, 0.111, 0.094, 0.064, 0.04, 0.029]
# }
season_data = {
    "season"                 : ["Winter", "Spring", "Summer", "Autumn"],
    "seasonal yield"         : [0.0383, 0.1103, 0.1177, 0.0660],
    "Seasonal tariff (€/kWh)": [0.4,    0.3,    0.25,   0.3],
    "Days in season"         : [90,     92,       92,    91]
}
seasons = pd.DataFrame(season_data)
df4_seasons = seasons[["season", "seasonal yield", "Seasonal tariff (€/kWh)", "Days in season" ]]

In [23]:
# Fill null values in LandPrice with the mean
Neighborhoods = df1_demand[['POINT_X', 'POINT_Y']].to_numpy()
neighborhood_id = df1_demand["UniqueID"].to_numpy()
Chargers = df_chargers[['POINT_X', 'POINT_Y', "LandPrice", 'Grid_supply']].to_numpy()
charger_id = df_chargers["ID_unique"].to_numpy()

In [24]:
# Validate input data (correct columns)
if (df1_demand["Population"] < 0).any():
    print("Warning: Negative population values detected")

if (df_chargers["LandPrice"] <= 0).any():
    print("Warning: Non-positive land costs detected")

# Optional coordinate sanity checks
if (df1_demand["POINT_X"].isna().any() or df1_demand["POINT_Y"].isna().any()):
    print("Warning: Missing demand coordinates detected")

if (df_chargers["POINT_X"].isna().any() or df_chargers["POINT_Y"].isna().any()):
    print("Warning: Missing charger coordinates detected")

In [25]:
# Road network data- Load shapefile
shapefile_path = "EindhovenSpeedNetwork.shp" 
if not os.path.exists(shapefile_path):
    raise FileNotFoundError(f"Shapefile not found at {shapefile_path}")
gdf = gpd.read_file(shapefile_path)
if gdf.crs.to_epsg() != 28992:
    gdf = gdf.to_crs(epsg=28992)

In [26]:
# CONFIG
DEFAULT_SPEED = 50.0
MIN_SPEED = 15.0
SNAP_TOLERANCE = None   # set to e.g. 0.5 (meters) to snap close coords, or None to disable

# 0) Ensure CRS (must be done earlier)
assert gdf.crs and gdf.crs.to_epsg() == 28992, "gdf must be EPSG:28992 (projected in meters)"

# 1) Preprocess speed column once
gdf["MAXSHD"] = pd.to_numeric(gdf["MAXSHD"], errors="coerce")
gdf["MAXSHD"] = gdf["MAXSHD"].fillna(DEFAULT_SPEED).clip(lower=MIN_SPEED)

# helper: optional snapping function to merge near-equal vertices (disable by setting SNAP_TOLERANCE=None)
def _snap(coord, tol):
    if tol is None:
        return (coord[0], coord[1])
    return (round(coord[0] / tol) * tol, round(coord[1] / tol) * tol)

# 2) Build graph
G = nx.Graph()
coord_to_id = {}
node_id = 0

for _, row in gdf.iterrows():
    geom = row.geometry
    speed_km_h = float(row["MAXSHD"])  # safe now

    lines = list(geom.geoms) if geom.geom_type == "MultiLineString" else [geom]
    for line in lines:
        coords = list(line.coords)
        for k in range(len(coords) - 1):
            s = _snap(coords[k], SNAP_TOLERANCE)
            e = _snap(coords[k+1], SNAP_TOLERANCE)

            if s not in coord_to_id:
                coord_to_id[s] = node_id
                G.add_node(node_id, geometry=Point(s), x=s[0], y=s[1])
                node_id += 1
            if e not in coord_to_id:
                coord_to_id[e] = node_id
                G.add_node(node_id, geometry=Point(e), x=e[0], y=e[1])
                node_id += 1

            u = coord_to_id[s]
            v = coord_to_id[e]

            dx = e[0] - s[0]
            dy = e[1] - s[1]
            distance_km = np.hypot(dx, dy) / 1000.0
            travel_time_hr = distance_km / speed_km_h

            if distance_km > 0 and travel_time_hr > 0:
                G.add_edge(u, v,
                           distance_km=distance_km,
                           travel_time_hr=travel_time_hr,
                           speed_km_h=speed_km_h,
                           weight=travel_time_hr)
            else:
                # very unlikely now; log and continue
                print(f"Warning: invalid edge {u}-{v} (d={distance_km}, t={travel_time_hr})")

G.graph['crs'] = 'EPSG:28992'

In [27]:
# Check graph connectivity
if not nx.is_connected(G):
    print("Step 5: Warning: Graph is not fully connected. Using largest connected component")
    largest_cc = max(nx.connected_components(G), key=len)
    G = G.subgraph(largest_cc).copy()

Step 5: Warning: Graph is not fully connected. Using largest connected component


In [28]:
# Create GeoDataFrames
gdf_neighborhoods = gpd.GeoDataFrame(df1_demand, geometry=gpd.points_from_xy(df1_demand['POINT_X'], df1_demand['POINT_Y']), crs='EPSG:28992')
gdf_chargers = gpd.GeoDataFrame(df_chargers, geometry=gpd.points_from_xy(df_chargers['POINT_X'], df_chargers['POINT_Y']), crs='EPSG:28992')

In [29]:
# Find nearest nodes
def find_nearest_node(point, graph):
    try:
        return ox.distance.nearest_nodes(graph, X=point.x, Y=point.y)
    except ValueError:
        print(f"Warning: No nearest node found for {point}")
        return None
neighborhood_nodes = [find_nearest_node(pt, G) for pt in gdf_neighborhoods.geometry]
charger_nodes = [find_nearest_node(pt, G) for pt in gdf_chargers.geometry]

In [30]:
# # Filter valid nodes
valid_neighborhood_idx = [i for i, n in enumerate(neighborhood_nodes) if n is not None]
valid_charger_idx = [j for j, c in enumerate(charger_nodes) if c is not None]

removed_neighborhoods = [i for i, n in enumerate(neighborhood_nodes) if n is None]
removed_chargers = [j for j, c in enumerate(charger_nodes) if c is None]

# Keep only valid entries (single consistent indexing)
neighborhood_nodes = [neighborhood_nodes[i] for i in valid_neighborhood_idx]
charger_nodes = [charger_nodes[j] for j in valid_charger_idx]

Neighborhoods = df1_demand.iloc[valid_neighborhood_idx].to_numpy()
neighborhood_id = df1_demand["UniqueID"].iloc[valid_neighborhood_idx].to_numpy()

Chargers = df_chargers.iloc[valid_charger_idx].to_numpy()
charger_id = df_chargers["ID_unique"].iloc[valid_charger_idx].to_numpy()

#Optional: print counts:
print(f"Removed neighborhoods: {len(removed_neighborhoods)}; Removed chargers: {len(removed_chargers)}")

In [31]:
# --- Step 1: Initialize arrays ---
num_neighborhoods = len(neighborhood_nodes)
num_chargers = len(charger_nodes)

travel_distances = np.full((num_neighborhoods, num_chargers), np.inf, dtype=np.float32)
travel_times = np.full((num_neighborhoods, num_chargers), np.inf, dtype=np.float32)

# --- Step 2: Efficient computation using single-source Dijkstra ---
for i, n_node in enumerate(neighborhood_nodes):
    # Compute travel times and distances from this neighborhood to all reachable nodes
    time_lengths = nx.single_source_dijkstra_path_length(G, n_node, weight="travel_time_hr")
    dist_lengths = nx.single_source_dijkstra_path_length(G, n_node, weight="distance_km")

    for j, c_node in enumerate(charger_nodes):
        # If charger is reachable from this neighborhood
        if c_node in time_lengths:
            distance = dist_lengths[c_node]
            # Optional max distance filter (keep only reachable within 1.5×4.4 km)
            # travel_times[i, j] = time_lengths[c_node]
            # travel_distances[i, j] = distance
            if distance <= 4.4 * 1.5:
                travel_distances[i, j] = distance
                travel_times[i, j] = time_lengths[c_node]
            # else remains np.inf

In [32]:
# start declarations and create sets
J = list()  # set of potential charger locations
I = list()  # set of demand locations
T = list()  # Set of solar yield times of the day
M = list()  # seasons
B = list()  # set of indices in cost ranges data

In [33]:
# declare parameters
ccp   = {}  # capacity of charger j at month m and time t
ccp_in= {}  # v2G capacity
dem   = {}  # demand at location i in month m in time t
gc    = {}  # Grid capacity available for EV consumption
lc    = {}  # Land cost at EVCS location
tt    = {}  # Travel time kWh
bess  = {}
annualised_bcost = {}
annualised_lc = {}
w     = {}

In [34]:
# declare decision variables
F   = {}    # objective function values
X   = {}    # amount of energy from charger j to demand location i in month m and time of the day t
S   = {}    # amount of energy from demand location i to station j in month m and time t
G   = {}    # Amount of surplus energy assigned to the charging station from the grid
Z   = {}    # Amount of energy from a charging station supplied to the grid.
Y   = {}    # binary... open new charger
BatType = {}    # Binary variable for battery type
INV = {}    # Inventory is the amount of energy stored at location j in time.

In [35]:
# declare constraints/equations
F = {}
E_dem = {}  # demand constraints
E_cbl = {}  # charger energy balance
E_gc  = {}  # Energy constraint from the grid
E_open= {}  # Energy constraint for opening the charger.
E_s   = {}  # Energy constraint for Supply to charger j
E_z   = {}  # Energy constraint for energy leaving the charger location to the grid
E_g   = {}  # energy constraint for energy assigned to charger j from grid.
E_x   = {}  # charger capacity to demand
E_INV = {}  # Available energy that can be stored. 
c_grid={}   # Energy consumption for the daily and seasonal tariffs.

In [36]:
# demand locations
for i in range(len(neighborhood_id)):
    I.append(i)

In [37]:
# charger locations
for j in range(len(charger_id)):
    J.append(j)

In [38]:
#  times of the day
for t in range(len(df3_time["Time"])):
    T.append(t)

In [39]:
# seasons of the year from 1 to 12
for m in range(len(df4_seasons["season"])):
    M.append(m)

In [40]:
for b in range(len(df5_Batcost["ID"])):
    B.append(b)

In [41]:
# initialization of demand# Demand calacualted
for i in I:
    for m in M:
        for t in T:
            Value = ((df1_demand.loc[i, 'Demand(kWh)']/1000) *  df3_time.loc[t, "Daily Demand variation"]) - ((((df1_demand.loc[i, "Solar_Supply_MW"])) * df3_time.loc[t, "time_interval"]) *   #convert to mwh. 
                (df3_time.loc[t, "Daily Yield"] * df4_seasons.loc[m, "seasonal yield"]))
            dem[i, m, t] = 0 if Value < 0 else Value         

In [42]:
# # initialization of travel time tt calculated and Pre-allocate tt with inf
tt = np.full((num_neighborhoods, num_chargers), np.inf, dtype=np.float32) # sets the numpy array with Inf initially, then the next line populates this with values, and inf division remains inf. Inf implies impossible or avoid at all cost

for i in I:
    for j in J:
        dist = travel_distances[i, j]
        time_val = travel_times[i, j]
        if dist > 0 and not np.isinf(dist) and not np.isinf(time_val):
            tt[i, j] = time_val / (av_EC * dist)
        # Invalid: stays inf

In [43]:
#initialization of gc_grid energy consumption
for j in J:
    for t in T:
            gc[j,t] = df2_chargers.loc[j, 'Grid_supply'] * df3_time.loc[t, "time_interval"] #Keep in mwh not kWh for scalling peurposes otherwisde * 1000 # Convert energy from mWh to kWh for 3hour time interval.(Maybe it should be three-hour intervals)

In [44]:
# initialization of charger capacity
for j in J:
    for t in T:
        ccp[j,t] = (22/1000) * df3_time.loc[t, "time_interval"] #convert to mW for charger capacity#charger capacity is for the 22kw type 2 charger recommended in the EU. This capacity is multiplied by a 3-hour interval to convert i to Energy.
        #ccp[j,t]=0.3e1  ###Change this to the equation above when we run with the total chargers.

In [45]:
#initialization for grid feed-in
for j in J:
    for t in T:
        ccp_in[j,t] = (9/1000) * df3_time.loc[t, "time_interval"]

In [46]:
#initialization of cost per kWh
for j in J:
   for b in B:
       bess[j,b] = (df5_Batcost.loc[b, "upper(kWh)"])/1000 #convert to mWh for scalling purposes

In [47]:
# --- Annualised cost initialisation ---
def CRF(r, N):
    return r / (1 - (1 + r) ** (-N))

def pv_battery_unit_cost(c_per_MWh, r, N_bat, T):
    n_replacements = (T - 1) // N_bat + 1
    return sum(c_per_MWh / (1 + r) ** (k * N_bat) for k in range(n_replacements))

r = 0.04
Tp = 30
N_bat = 10
CRF_T = CRF(r, Tp)

for j in J:
    for b in B:
        raw_cost_per_MWh = df5_Batcost.loc[b, "Cost/kWh"] * 1000  # €/MWh
        # PV of battery over horizon
        pv_unit_cost = pv_battery_unit_cost(raw_cost_per_MWh, r, N_bat, Tp)  # €/MWh
        # Annualised per kWh-year
        annualised_bcost[j,b] = pv_unit_cost * CRF_T  # €/MWh·year

In [48]:
# # Adjusted battery cost dictionary
# for j in J:
#     for b in B:
#         raw_cost_per_MWh = df5_Batcost.loc[b, "Cost/kWh"] * 1000 # convert Euros to eurosK and kWh to mWh in this case value is the same in ek/mwh with e/kwh
#         # PV of battery over horizon
#         pv_unit_cost = pv_battery_unit_cost(raw_cost_per_MWh, r, N_bat, Tp)
#         # Annualised per kWh-year
#         annualised_bcost[j,b] = (pv_unit_cost * CRF_T)

In [49]:
# Adjusted land cost dictionary
for j in J:
    raw_land_cost = df2_chargers.loc[j, "LandPrice"] * 1000 # increased land prices for scaling with other units in the model because they were too small
    pv_land = raw_land_cost  # one-shot, no replacement
    annualised_lc[j] = (pv_land * CRF_T)

In [50]:
#normalizing the weighting of the seasonal tariff and then the C-grid
sum_S = (df4_seasons["Seasonal tariff (€/kWh)"].sum())

s_factor = {}
for m in M:
    s_factor[m] = df4_seasons.loc[m, "Seasonal tariff (€/kWh)"]/ sum_S

# --- Combine to form season × time tariffs (€/MWh) ---
for m in M:
    for t in T:
        c_grid[m, t] = ((df3_time.loc[t, "Daily tariff(€/kWh)"] * s_factor[m]) * 1000) #creates a grid consumption tarrif daily influenced by season and daily tarrif. convert also to mwh

In [51]:
#cost variation of the Grid supply(chekced and correctd per year)
# for m in M:
#     for t in T:
#         w[m,t]= (df4_seasons.loc[m, "Days in season"] * df3_time.loc[t, "time_interval"]) #convert to year so that it is consistant with anualization
for m in M:
    w[m]= df4_seasons.loc[m, "Days in season"] #using days only in order to convert the G to annual consumption because gc already converts MW to MWh

In [52]:
os.environ["GRB_LICENSE_FILE"] = "gurobi.lic"
env = gp.Env(empty=True)
env.start()

md = gp.Model("EVCS_optimization", env=env)
md.setParam("NodefileStart", 1.5)
md.setParam("NodefileDir", "/tmp/grb_nodes")
md.setParam("MIPGap", 0.00025)

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2727022
Academic license 2727022 - for non-commercial use only - registered to mo___@wur.nl
Set parameter NodefileStart to value 1.5
Set parameter NodefileDir to value "/tmp/grb_nodes"
Set parameter MIPGap to value 0.00025


DECISSION VARIABLES X, S AND Z, G

In [53]:
# validity filters
valid_X = {(j, i, m, t) for j in J for i in I for m in M for t in T if travel_distances[i, j] <= 4.4 * 1.5}
valid_S = {(i, j, m, t) for i in I for j in J for m in M for t in T if travel_distances[i, j] <= 4.4 * 1.5}

# variable creation
# X = {k: md.addVar(name=f"X{k}", vtype=GRB.CONTINUOUS) for k in valid_X}
# S = {k: md.addVar(name=f"S{k}", vtype=GRB.CONTINUOUS) for k in valid_S}
X = {k: md.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name=f"X{tuple(k)}") for k in valid_X}
S = {k: md.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name=f"S{tuple(k)}") for k in valid_S}

In [54]:
# for j in J:
#     for m in M:
#         for t in T:
#             Z[j, m, t] = md.addVar(name="Z({},{},{})".format(j, m, t), vtype=GRB.CONTINUOUS)
#             G[j, m, t] = md.addVar(name="G({},{},{})".format(j, m, t), vtype=GRB.CONTINUOUS)

# Station-level variables
for j in J:
    for m in M:
        for t in T:
            Z[j, m, t] = md.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name=f"Z({j},{m},{t})")
            G[j, m, t] = md.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name=f"G({j},{m},{t})")
            INV[j, m, t] = md.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name=f"INV({j},{m},{t})")


In [55]:
# #3 Decision variables for I inventory what is possible to be stored
# for j in J:
#     for m in M:
#         for t in T:
#             INV[j,m,t] = md.addVar(name="INV({},{},{})".format(j, m, t), vtype=GRB.CONTINUOUS)

In [56]:
# === Binary activation and battery type ===
for j in J:
    Y[j] = md.addVar(vtype=GRB.BINARY, name="Y({})".format(j))

In [57]:
for j in J:
    for b in B:
        BatType[j, b] = md.addVar(vtype=GRB.BINARY, name="BatType({},{})".format(j, b))

THE OBJECTIVE FUNCTION: LAND COST AND TOTAL TRAVEL TIME

In [58]:
# objective function variables initialized(checked and corrected)
F["LC"] =md.addVar(name="F(LC)", vtype=GRB.CONTINUOUS, lb=0)
F["TT"] =md.addVar(name="F(TT)", vtype=GRB.CONTINUOUS, lb=0)
F["BC"] =md.addVar(name="F(BC)", vtype=GRB.CONTINUOUS, lb=0)
F["EC"] =md.addVar(name="F(EC)", vtype=GRB.CONTINUOUS, lb=0)

CONSTRAINTS

In [59]:
# checked and correct
unreachable = [i for i in range(num_neighborhoods)
               if all(np.isinf(travel_distances[i, j]) for j in range(num_chargers))]
for i in unreachable:
    for m in M:
        for t in T:
            dem[i,m,t] = 0

In [60]:
#1=X # #add X constraint with Y[j] (Checked and is correct)
for j in J:
    for m in M:
        for t in T:
            E_x[j, m, t] = md.addConstr(gp.quicksum(X[j, i, m, t] for i in I if (j, i, m, t) in valid_X) <= ccp[j, t] * Y[j], name=f"E_x({j},{m},{t})")

In [61]:
#2=S # initialization of the big M for the charging station(Checked and corrected)## think about the valid_s expression
for j in J:
    for m in M:
        for t in T:
            E_s[j,m,t]= md.addConstr(gp.quicksum(S[i, j, m, t] for i in I if (i, j, m, t) in valid_S) <= ccp_in[j,t] * Y[j], name=f"E_s({j},{m},{t})")  # formulate the ccpiin parameter.

In [62]:
#3=G Define BigMg per j,t (aggregate over b; assume B is the set of batteries)(checked and corrected)
for j in J:
    for m in M:
        for t in T:
            md.addConstr(G[j, m, t] <= gc[j, t] * Y[j], name=f"E_g({j},{m},{t})")# included but check fotr gc indices

In [63]:
#4=Z Initialization of the big M for the charger to the Grid energy supply: constraints for supply to the grid (#Checked and corrected)
for j in J:
    for m in M:
        for t in T:
            md.addConstr(Z[j, m, t] <= gc[j,t] * Y[j], name=f"Z_max({j},{m},{t})") #since one battery type is selected for bn B ius correct but can also be left outNonlinear????????

In [64]:
#5= INV  
for j in J:
    for m in M:
        for t in T: 
            md.addConstr(INV[j, m, t] <= gp.quicksum(BatType[j, b] * bess[j, b] for b in B), name="Cap_def({},{},{})".format(j, m, t))

In [65]:
#6= X-S #energy Balance X, S, and Dem(Checked and corrected)
for i in I:
    for m in M:
        for t in T:
            E_dem[i, m, t] = md.addConstr(gp.quicksum(X[j, i, m, t] for j in J if (j, i, m, t) in valid_X) - gp.quicksum(S[i, j, m, t] for j in J if (i, j, m, t) in valid_S) == dem[i, m, t], name=f"E_dem({i},{m},{t})")


In [66]:
#8 INV #Inventory constraints: (checked and corrected)
T_last = max(T)

for j in J:
    for m in M:
        for t in T:
            if t == 0:
                # use last-period inventory as previous
                E_INV[j, m, t] = md.addConstr( INV[j, m, t] == INV[j, m, T_last] + gp.quicksum(S[i, j, m, t] for i in I if (i, j, m, t) in valid_S) + G[j, m, t]- Z[j, m, t] 
                                               - gp.quicksum(X[j, i, m, t] for i in I if (j, i, m, t) in valid_X), name=f"E_INV({j},{m},{t})")
            else:
                E_INV[j, m, t] = md.addConstr(INV[j, m, t] == INV[j, m, t - 1] + gp.quicksum(S[i, j, m, t] for i in I if (i, j, m, t) in valid_S) + G[j, m, t] - Z[j, m, t] 
                                              - gp.quicksum(X[j, i, m, t] for i in I if (j, i, m, t) in valid_X), name=f"E_INV({j},{m},{t})")

In [67]:
for j in J:
    md.addConstr(gp.quicksum(BatType[j,b] for b in B) == Y[j], name="Link_Bat_to_EVS_activation({})".format(j)) # Station activation implies battery activation. Model selects which battery type 

In [68]:
#objective functions
F["EC"] = gp.quicksum(c_grid[m,t] * G[j,m,t] * w[m] for j in J for m in M for t in T) # w[m] only days in a season because G is already in hours of day.
F["LC"] = gp.quicksum(annualised_lc[j] * Y[j] for j in J)
F["BC"] = gp.quicksum(bess[j,b] * annualised_bcost[j,b] * BatType[j,b] for j in J for b in B)
F["TT"] = gp.quicksum(((S[i,j,m,t] if (i,j,m,t) in valid_S else 0) + (X[j,i,m,t] if (j,i,m,t) in valid_X else 0)) * tt[i,j] for i in I for j in J for m in M for t in T if np.isfinite(tt[i,j]))

In [69]:
#Objective for Annual energy consumption cost was c_grid is the cost of energy per m and t, and w is the number of hours in a year, and G energy from the  grid.
#EC_fob["EC"] = md.addConstr(F["EC"] == gp.quicksum(c_grid[m, t] * G[j, m, t] * w[m, t] for j in J for m in M for t in T), name="E_fob_EC")

In [70]:
# Define the objective function for land costs
#E_fob["LC"] = md.addConstr(F["LC"] == gp.quicksum(annualised_lc[j] * Y[j] for j in J), "E_F(LC)")

In [71]:
# # Define the objective function of total travel time
#T_fob["TT"] = md.addConstr(F["TT"] == gp.quicksum(((S[i, j, m, t] if (i, j, m, t) in valid_S else 0) + (X[j, i, m, t] if (j, i, m, t) in valid_X else 0)) * tt[i, j] for i in I for j in J for m in M for t in T if np.isfinite(tt[i,j])
    #), name="T_fob_TT")

In [72]:
# Define objective for battery storage cost
#B_fob["BC"] = md.addConstr(F["BC"] == gp.quicksum(bess[j,b] * annualised_bcost[j,b] * BatType[j,b] for j in J for b in B), name="B_fob_BC")

In [73]:
print(os.getcwd())

/home/WUR/katon002


In [ ]:
# === Phase 1: minimize travel time ===
md.setObjective(F["TT"], GRB.MINIMIZE)
md.optimize()

if md.SolCount == 0:
    raise RuntimeError("Phase 1 failed — no feasible solution found.")

fixed_val = md.ObjVal

# === Phase 2: lock travel time and minimize cost ===
md.resetParams()
md.setParam("MIPFocus", 2)            
md.setParam("MIPGap", 0.00025)          
md.setParam("NodefileStart", 1.5)
md.setParam("NodefileDir", "/tmp/grb_nodes")

tol = 5e-3
md.addConstr(F["TT"] <= fixed_val * (1 + tol), name="Fix_TT")
md.setObjective(F["LC"] + F["BC"] + F["EC"], GRB.MINIMIZE)
md.optimize()

if md.SolCount == 0:
    raise RuntimeError("Phase 2 failed — no feasible solution found.")

# === After Phase 2 optimization ===
print("Final Status:", md.Status, "Gap:", md.MIPGap, "Obj:", md.ObjVal)

# Register all symbolic variable names *before* writing any files
# md.update()



In [90]:
print("Fix_TT slack:", md.getConstrByName("Fix_TT").Slack)  # in TT→MC model

Fix_TT slack: 8.731149137020111e-11


In [89]:

# Write model and solution
md.write("TT_model.mps")      # exact numerical model (lossless vs. LP)
md.write("TT_model.prm")      # all parameter settings
md.write("TT_solution.sol")   # final variable values

# === Write text summary ===
OPEN_THR = 5e-5
# with open("final_summary.txt", "w") as f:
#     f.write("=== Final Two-Phase Model Summary ===\n")
#     f.write(f"Phase1_Obj={fixed_val}\n")
#     f.write(f"Phase2_Obj={md.ObjVal}\n")
#     f.write(f"BestBd={md.ObjBound}\n")
#     f.write(f"Gap={md.MIPGap}\n")
#     f.write(f"Runtime={md.Runtime}\n")
#     f.write(f"Chargers_Opened={sum(1 for j in J if Y[j].X > OPEN_THR)} / {len(J)}\n")

LC_val = sum(annualised_lc[j] * Y[j].X for j in J)
BC_val = sum(bess[j,b] * annualised_bcost[j,b] * BatType[j,b].X for j in J for b in B)
EC_val = sum(c_grid[m,t] * G[j,m,t].X * w[m] for j in J for m in M for t in T)
TT_X = sum(tt[i,j] * X[j,i,m,t].X for (j,i,m,t) in valid_X if np.isfinite(tt[i,j]))
TT_S = sum(tt[i,j] * S[i,j,m,t].X for (i,j,m,t) in valid_S if np.isfinite(tt[i,j]))
TT_val = TT_X + TT_S

print("Objective Values:")
print(f"  Land Cost (LC):   {LC_val:.2f} €")
print(f"  BESS Cost (BC):   {BC_val:.2f} €")
print(f"  Energy Cost (EC): {EC_val:.2f} €")
print(f"  Travel Time (TT): {TT_val:.2f}")
print(f"  Total Cost (LC+BC+EC): {LC_val + BC_val + EC_val:.2f} €\n")

total_chargers = sum(1 for j in J if Y[j].X > OPEN_THR)
print(f"Total Chargers Opened: {total_chargers}")

# print("\nSummary of Nonzero Variables:")
# print(f"  Y: {sum(1 for v in Y.values() if v.X > OPEN_THR)}/{len(Y)} nonzero")
# for name, d in [("X", X), ("S", S), ("G", G), ("Z", Z), ("INV", INV)]:
#     n_nonzero = sum(1 for v in d.values() if abs(v.X) > 1e-9)
#     print(f"  {name}: {n_nonzero}/{len(d)} nonzero")

Objective Values:
  Land Cost (LC):   19601815.22 €
  BESS Cost (BC):   6866319.27 €
  Energy Cost (EC): 6018402.88 €
  Travel Time (TT): 139960.52
  Total Cost (LC+BC+EC): 32486537.38 €

Total Chargers Opened: 1071


In [76]:
#Final summary (text)
OPEN_THR = 5e-5
with open("TT_run_cost_summary.txt", "w") as f:
    f.write("=== Final Two-Phase_TT_run Model Summary ===\n")
    #f.write(f"Phase1_Obj={phase1_obj}\n")          # store this right after phase 1
    f.write(f"TT_Phase2_Obj={md.ObjVal}\n")
    f.write(f"TT_BestBd={md.ObjBound}\n")
    f.write(f"TT_Gap={md.MIPGap}\n")
    f.write(f"TT_Runtime={md.Runtime}\n")
    f.write(f"TT_Chargers_Opened={sum(1 for j in J if Y[j].X > OPEN_THR)} / {len(J)}\n")

In [77]:
#Nonzero counts
nz = []
nz.append(("Y", sum(1 for j in J if Y[j].X > OPEN_THR), len(J)))
nz.append(("X", sum(1 for v in X.values() if abs(v.X) > 1e-9), len(X)))
nz.append(("S", sum(1 for v in S.values() if abs(v.X) > 1e-9), len(S)))
nz.append(("G", sum(1 for v in G.values() if abs(v.X) > 1e-9), len(G)))
nz.append(("Z", sum(1 for v in Z.values() if abs(v.X) > 1e-9), len(Z)))
nz.append(("INV", sum(1 for v in INV.values() if abs(v.X) > 1e-9), len(INV)))
nz.append(("BatType", sum(1 for v in BatType.values() if abs(v.X) > 1e-9), len(BatType)))

df_nz = pd.DataFrame(nz, columns=["var", "nonzero", "total"])
df_nz.to_csv("TT_nonzero_counts_TT_run_.csv", index=False)


In [78]:
# Siting results (open stations with coordinates)
df_chargers = pd.DataFrame(datacharger)  # or pd.read_csv(...)

open_rows = []
for j in J:
    if Y[j].X > OPEN_THR:
        # find chosen battery (maybe none)
        chosen_b = [b for b in B if BatType[j, b].X > 0.5]
        open_rows.append({
            "j": j,
            "ID_unique": df_chargers.loc[j, "ID_unique"],
            "POINT_X": df_chargers.loc[j, "POINT_X"],
            "POINT_Y": df_chargers.loc[j, "POINT_Y"],
            "Y": Y[j].X,
            "LandPrice": df_chargers.loc[j, "LandPrice"],
            "Grid_supply": df_chargers.loc[j, "Grid_supply"],
            "Actual_required_MW": df_chargers.loc[j, "Actual required at location(MW)"],
            "Available_intake_MW": df_chargers.loc[j, "Available Grid intake(Feed-in)(MW)"],
            "Chosen_BatTypes": ",".join(str(b) for b in chosen_b)
        })

df_open = pd.DataFrame(open_rows)
df_open.to_csv("TT_run_sites_selected_.csv", index=False)

In [79]:
#Objective values results
LC_val = sum(annualised_lc[j] * Y[j].X for j in J)
BC_val = sum(bess[j,b] * annualised_bcost[j,b] * BatType[j,b].X for j in J for b in B)
EC_val = sum(c_grid[m,t] * G[j,m,t].X * w[m] for j in J for m in M for t in T)
TT_X = sum(tt[i,j] * X[j,i,m,t].X for (j,i,m,t) in valid_X if np.isfinite(tt[i,j]))
TT_S = sum(tt[i,j] * S[i,j,m,t].X for (i,j,m,t) in valid_S if np.isfinite(tt[i,j]))
TT_val = TT_X + TT_S

df_obj = pd.DataFrame([{
    "LC": LC_val,
    "BC": BC_val,
    "EC": EC_val,
    "TT": TT_val,
    "Total_cost": LC_val + BC_val + EC_val
}])
df_obj.to_csv("TT_objective_breakdown_TT.csv", index=False)

In [80]:
#Demand → station assignments
rows = []
for (j, i, m, t), var in X.items():
    if var.X > 1e-9:
        rows.append({
            "demand_i": i,
            "station_j": j,
            "m": m,
            "t": t,
            "flow_X": var.X,
            "travel_time": tt[i, j] if np.isfinite(tt[i, j]) else None
        })

df_assign = pd.DataFrame(rows)
df_assign.to_csv("TT_assignment_summary.csv", index=False)

In [81]:
#Time-series per station (optional)
lowz = []
for (j, m, t), v in G.items():
    if v.X > 1e-9:
        lowz.append({
            "station_j": j,
            "m": m,
            "t": t,
            "G": v.X,
            "Z": Z[j, m, t].X if (j, m, t) in Z else None,
            "INV": INV[j, m, t].X if (j, m, t) in INV else None,
        })

df_ts = pd.DataFrame(lowz)
df_ts.to_csv("TT_station_timeseries.csv", index=False)

RESULTS PRINTING FIXED COST

In [82]:
# 🧩 2️⃣ Energy Flow Network (X, S, G, Z, INV)
# Purpose: Quantify total energy movement across the system.
# === ENERGY FLOW NETWORK ===
df_flow = pd.DataFrame([
    (i, j, m, t, X[j,i,m,t].X, S[i,j,m,t].X if (i,j,m,t) in S else 0, 
     G[j,m,t].X, Z[j,m,t].X, INV[j,m,t].X)
    for (j,i,m,t) in valid_X
], columns=["demand_i","station_j","m","t","flow_X","flow_S","grid_in_G","grid_out_Z","inventory_INV"])

df_flow["total_in"]  = df_flow["grid_in_G"] + df_flow["flow_S"]
df_flow["total_out"] = df_flow["grid_out_Z"] + df_flow["flow_X"]
df_flow.to_csv("TT_energy_network.csv", index=False)
#print(df_flow.head())

In [83]:
# 🧩 3️⃣ Temporal Dynamics (m, t)
# Purpose: Aggregate flows by season and hour for grid/EV/storage dynamics.
# === TEMPORAL ENERGY PROFILE ===
df_temporal = df_flow.groupby(["m","t"], as_index=False)[["grid_in_G","grid_out_Z","inventory_INV", "flow_S","flow_X"]].sum()
df_temporal["net_grid"] = df_temporal["grid_in_G"] - df_temporal["grid_out_Z"]
df_temporal.to_csv("TT_temporal_dynamics.csv", index=False)
#print(df_temporal.head())


In [84]:
# 🧩 4️⃣ Travel Time & Accessibility
# Purpose: Measure user access improvements, average and distribution.
# === TRAVEL TIME AND ACCESSIBILITY ===
df_tt = pd.DataFrame([
    (i, j, m, t, X[j,i,m,t].X, tt[i,j])
    for (j,i,m,t) in valid_X if X[j,i,m,t].X > 0
], columns=["demand_i","station_j","m","t","flow_X","travel_time"])

# Average travel time weighted by flow
avg_tt = (df_tt["flow_X"] * df_tt["travel_time"]).sum() / df_tt["flow_X"].sum()
print(f"Weighted average travel time: {avg_tt:.2f} h")

df_tt.to_csv("TT_travel_times.csv", index=False)


Weighted average travel time: inf h


In [85]:
# 🧩 5️⃣ Cost Component Breakdown (LC, BC, EC, TT)
# Purpose: Extract the components contributing to the objective.
# === COST COMPONENTS ===
LC_val = sum(annualised_lc[j] * Y[j].X for j in J)
BC_val = sum(bess[j,b] * annualised_bcost[j,b] * BatType[j,b].X for j in J for b in B)
EC_val = sum(c_grid[m,t] * G[j,m,t].X * w[m] for j in J for m in M for t in T)
TT_X = sum(tt[i,j] * X[j,i,m,t].X for (j,i,m,t) in valid_X if np.isfinite(tt[i,j]))
TT_S = sum(tt[i,j] * S[i,j,m,t].X for (i,j,m,t) in valid_S if np.isfinite(tt[i,j]))
TT_val = TT_X + TT_S

df_cost = pd.DataFrame({
    "LC_k€":[LC_val],
    "BC_k€":[BC_val],
    "EC_k€":[EC_val],
    "TT_h":[TT_val],
    "Total_k€":[LC_val+BC_val+EC_val]
})
df_cost.to_csv("TT_cost_breakdown.csv", index=False)
#print(df_cost)

/tmp/ipykernel_681062/824983143.py:7: RuntimeWarning: invalid value encountered in scalar multiply
  TT_val = sum(tt[i,j] * X[j,i,m,t].X for (j,i,m,t) in valid_X)


In [86]:
# 🧩 6️⃣ Charger Utilization & Assignment (X)
# Purpose: Quantify how many chargers serve each demand node and total energy served per charger.
# === CHARGER UTILIZATION ===
df_X = df_assign[["demand_i","station_j","flow_X","travel_time"]].copy()

# Chargers per demand node
df_demand_links = df_X.groupby("demand_i")["station_j"].nunique().reset_index()
df_demand_links.columns = ["demand_i","n_chargers"]

# Energy per station
df_station_load = df_X.groupby("station_j")["flow_X"].sum().reset_index()
df_station_load.columns = ["station_j","total_energy_MWh"]

df_demand_links.to_csv("TT_demand_links.csv", index=False)
df_station_load.to_csv("TT_station_utilization.csv", index=False)
#print(df_station_load)

In [87]:
# 🧩 7️⃣ Storage Activity (INV, BatType)
# Purpose: Show which stations have batteries and their size/type.
# === STORAGE ACTIVITY ===
df_storage = pd.DataFrame([
    (j, b, BatType[j,b].X)
    for j in J for b in B if BatType[j,b].X > 0
], columns=["station_j","battery_type","active"])

df_storage["battery_type_label"] = df_storage["battery_type"].map({0:"Small",1:"Medium",2:"Large"})

# Inventory pattern
df_inv_series = pd.DataFrame([
    (j, m, t, INV[j,m,t].X)
    for j in J for m in M for t in T if INV[j,m,t].X > 0
], columns=["station_j","m","t","inventory_MWh"])

df_storage.to_csv("TT_storage_sites.csv", index=False)
df_inv_series.to_csv("TT_inventory_timeseries.csv", index=False)
#print(df_inv_se)

In [88]:
# 🧩 9️⃣ Model Performance Summary
# Purpose: Document solver efficiency and reproducibility.
# === MODEL PERFORMANCE ===
df_perf = pd.DataFrame({
    "Objective": [md.ObjVal],
    "BestBound": [md.ObjBound],
    "Gap": [md.MIPGap],
    "Runtime_s": [md.Runtime],
    "Vars_total": [md.NumVars],
    "Constr_total": [md.NumConstrs],   # updated name
    "Chargers_opened": [sum(1 for j in J if Y[j].X > OPEN_THR)]
})

df_perf.to_csv("TT_model_performance.csv", index=False)
#print(df_perf)
